# Lesson 1: Router Engine

Build a simple agentic RAG router that chooses between **summarization** and **vector Q&A** over a single PDF, using **Gemini** via LlamaIndex.

## Setup

Uses `GOOGLE_API_KEY` from your project `.env` (same pattern as week_1 / week_2).

Install deps once from this folder:

```bash
pip install -r requirements.txt
```

Place `metagpt.pdf` in this `week_3/` directory (download tip below).

In [1]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()
print(f"LLM: Gemini | Embeddings (local): {embed_model.model_name}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LLM: Gemini | Embeddings (local): BAAI/bge-small-en-v1.5


In [2]:
import nest_asyncio

nest_asyncio.apply()

## Load Data

To download the MetaGPT paper:

```bash
# from the week_3 directory
curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf
```

Or use the cell below.

In [3]:
# !curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf

from helper import load_pdf

# Important: use PDFReader (via load_pdf), not raw SimpleDirectoryReader —
# otherwise the PDF binary is treated as text and explodes into thousands of chunks.
documents = load_pdf("metagpt.pdf")
print(f"Loaded {len(documents)} pages")


Loaded 29 pages


## Define LLM and Embedding model

`configure_settings()` sets **Gemini for the LLM** and a **local HuggingFace embedding model** (no embedding API quota). Below we chunk the document.

In [4]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)
print(f"Created {len(nodes)} nodes")

Created 34 nodes


In [5]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from helper import DEFAULT_LLM_MODEL, DEFAULT_EMBED_MODEL, get_embed_model

# Explicit (same as configure_settings) — useful when teaching
Settings.llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY)
Settings.embed_model = get_embed_model(DEFAULT_EMBED_MODEL)  # local, free


2026-07-13 21:59:33,894 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-13 21:59:34,232 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-13 21:59:34,251 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-13 21:59:34,520 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-13 21:59:34,540 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-13 21:59:34,544 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-13 21:59

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-13 21:59:36,854 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 21:59:37,130 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 21:59:37,435 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 21:59:37,721 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 21:59:38,012 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-13 21:59:38,041 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

## Define Summary Index and Vector Index over the Same Data

In [6]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

## Define Query Engines and Set Metadata

In [7]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,  # sequential calls play nicer with low RPM quotas
)
vector_query_engine = vector_index.as_query_engine()


In [8]:
from llama_index.core.tools import QueryEngineTool

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarization questions related to MetaGPT",
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific context from the MetaGPT paper.",
)

## Define Router Query Engine

In [9]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True,
)

In [10]:
response = query_engine.query("What is the summary of the document?")
print(str(response))

2026-07-13 21:59:49,972 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 21:59:58,905 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-13 21:59:58,908 - INFO - Selecting query engine 0: The question asks for a summary of the document, and choice (1) explicitly states it is useful for summarization questions..
2026-07-13 21:59:58,931 - ERROR - Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108170d80> is already entered
2026-07-13 21:59:58,932 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 0: The question asks for a summary of the document, and choice (1) explicitly states it is useful for summarization questions..


2026-07-13 22:00:29,933 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


MetaGPT is a meta-programming framework for multi-agent collaboration based on Large Language Models (LLMs) that incorporates human-like Standardized Operating Procedures (SOPs) to streamline workflows and reduce errors caused by cascading hallucinations. The framework utilizes an assembly line paradigm, assigning specialized roles—such as Product Manager, Architect, Project Manager, Engineer, and QA Engineer—to break down complex software engineering tasks into manageable subtasks.

Key features of MetaGPT include:
*   **Structured Communication:** Instead of relying solely on natural language dialogue, agents generate structured outputs, including Product Requirements Documents (PRDs), design artifacts, flowcharts, and interface specifications.
*   **Communication Protocol:** To enhance efficiency and avoid information overload, MetaGPT employs a shared message pool and a publish-subscribe mechanism, allowing agents to access relevant information based on their role profiles.
*   **E

In [12]:
print(len(response.source_nodes))

34


In [13]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

2026-07-13 22:01:05,219 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 22:01:25,051 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-13 22:01:25,056 - INFO - Selecting query engine 1: The question asks for a specific mechanism ('How do agents share information'), which requires retrieving specific context from the MetaGPT paper rather than a general summary..


Selecting query engine 1: The question asks for a specific mechanism ('How do agents share information'), which requires retrieving specific context from the MetaGPT paper rather than a general summary..


2026-07-13 22:01:25,357 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 22:01:39,752 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


Agents share information through structured communication using documents and diagrams rather than dialogue. This is facilitated by a shared message pool where agents publish their structured messages and can transparently access messages from other entities. To prevent information overload, a subscription mechanism allows agents to use their role profiles and role-specific interests to select and extract only the relevant information they need.


## Let's put everything together

`utils.get_router_query_engine` wraps the steps above and uses Gemini by default.

In [14]:
from utils import get_router_query_engine

query_engine = get_router_query_engine("metagpt.pdf")

2026-07-13 22:01:50,812 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-13 22:01:51,240 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-13 22:01:51,264 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-13 22:01:51,647 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-13 22:01:51,674 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-13 22:01:51,676 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-13 22:01

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-13 22:01:53,990 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 22:01:54,314 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 22:01:54,585 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 22:01:54,867 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-13 22:01:55,148 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-13 22:01:55,171 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [15]:
response = query_engine.query("Tell me about the ablation study results?")
print(str(response))

2026-07-13 22:02:01,683 - INFO - AFC is enabled with max remote calls: 10.
2026-07-13 22:02:12,047 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-13 22:02:12,051 - INFO - Selecting query engine 1: The question asks for specific results from an ablation study, which requires retrieving specific context from the MetaGPT paper rather than a general summarization..
2026-07-13 22:02:12,146 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 1: The question asks for specific results from an ablation study, which requires retrieving specific context from the MetaGPT paper rather than a general summarization..


2026-07-13 22:02:47,530 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


The results regarding the impact of instruction levels and the use of different LLM backends are as follows:

**Impact of Instruction Levels**
Experiments comparing high-level prompts (e.g., "Create a brick breaker game") with detailed prompts (which provide specific game mechanics) revealed that:
*   **Detailed prompts** lead to better software projects with lower productivity ratios (118.0 compared to 163.8 for high-level prompts) due to clearer functions and requirements. Detailed prompts achieved an executability rating of 4.0.
*   **High-level prompts** still generate "good enough" software with an executability rating of 3.8, which is comparable to the detailed prompt scenario.

**LLM Backend Performance**
When using different LLMs as agent backends for MetaGPT on SoftwareDev, the performance varied:
*   **GPT-4:** Achieved the highest executability score of 3.8 with 1.2 revisions.
*   **GPT-3.5:** Achieved an executability score of 2.8 with 2.4 revisions.
*   **Deepseek Coder 33